In [ ]:
from google.cloud import storage
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

bucket_name = 'medical_insurance_chatbot'

storage_client = storage.Client()

try:
    logging.info(f"Attempting to list top-level folders (prefixes) in bucket: {bucket_name}")

    # Use delimiter='/' to list objects at the top level and get common prefixes (folders)
    iterator = storage_client.list_blobs(bucket_name, delimiter='/')

    # Access the prefixes attribute by iterating through pages
    top_level_prefixes = set()
    for page in iterator.pages:
        if page.prefixes:
            top_level_prefixes.update(page.prefixes)

    if top_level_prefixes:
        print("\nTop-level folders (prefixes) found:")
        for prefix in top_level_prefixes:
            print(prefix)
    else:
        print("\nNo top-level folders (prefixes) found in the bucket.")
        logging.warning("No folder prefixes found. Check bucket contents.")

except Exception as e:
    logging.error(f"An error occurred during folder listing: {e}")
    print(f"\nCould not list folders in bucket '{bucket_name}'. Please check:")
    print(f"- The bucket name is correct.")
    print(f"- Your Google Cloud credentials and permissions are set up correctly.")
    print(f"- The service account/user running this code has 'Storage Object Viewer' role or equivalent.")
    top_level_prefixes = []  # Ensure prefixes list is empty if listing fails

# --- Code to list files within each folder ---
print("\nListing files within each found folder:")

if top_level_prefixes:
    for folder_prefix in top_level_prefixes:
        logging.info(f"Listing blobs in prefix: {folder_prefix}")

        try:
            # List blobs *within* this specific folder prefix.
            # No delimiter here unless you want to list sub-sub-folders.
            folder_iterator = storage_client.list_blobs(bucket_name, prefix=folder_prefix)

            # Iterate directly over the blobs within this specific prefix
            folder_blobs_found = False
            print(f"--- Files in folder: {folder_prefix} ---")
            for blob in folder_iterator:  # Directly iterate over the blobs
                print(blob.name)
                folder_blobs_found = True

            if not folder_blobs_found:
                print(f"No files found in folder: {folder_prefix}")
                logging.warning(f"No files found in expected folder: {folder_prefix}")

        except Exception as e:
            logging.error(f"An error occurred while listing files in {folder_prefix}: {e}")
            print(f"Could not list files in folder '{folder_prefix}'.")

else:
    print("Skipping file listing as no folders were found.")

print("\nFinished listing folders and files.")

In [ ]:
import os
import json  # Need json for saving
from PyPDF2 import PdfReader
from pdf2image import convert_from_path
import pytesseract
import sys

# Tesseract OCR setup (ensure Tesseract is installed on your system)
# Update this path if needed based on your Tesseract installation
# For Windows: r'C:\Program Files\Tesseract-OCR\tesseract.exe'
# For macOS: r'/usr/local/bin/tesseract' or similar path from `which tesseract`
# For Linux: r'/usr/bin/tesseract' or similar path from `which tesseract`
try:
    pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'
    print("Tesseract path set successfully.")
except Exception as e:
    print(f"Warning: Could not set Tesseract path. Check installation and path. Error: {e}", file=sys.stderr)
    print("Hint: If Tesseract is installed, find its executable path and update pytesseract.pytesseract.tesseract_cmd", file=sys.stderr)


# Configuration
INPUT_PARENT_FOLDER = "/home/jupyter/Medical_files"  # Update this path to your parent folder
OUTPUT_FOLDER = "/home/jupyter/Extracted_Text_v2" # Folder to save extracted text JSONs

# Create the output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Output folder ensured: {OUTPUT_FOLDER}")


def extract_text_from_pdf(file_path):
    """Extract text from a PDF file using PyPDF2."""
    print(f"Attempting PyPDF2 extraction for: {os.path.basename(file_path)}")
    try:
        reader = PdfReader(file_path)
        text = ""
        for page_num in range(len(reader.pages)):
             # Added page number for better debugging
            try:
                page = reader.pages[page_num]
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n" # Add newline between page texts
                else:
                    print(f"  PyPDF2: No text extracted from page {page_num + 1}", file=sys.stderr)
            except Exception as page_e:
                 print(f"  Error extracting text from page {page_num + 1} with PyPDF2: {page_e}", file=sys.stderr)
                 # Continue to next page even if one fails
                 pass # Or text += "[Error extracting page]"

        if text and text.strip():
            print(f"PyPDF2 extracted text (approx. length: {len(text.strip())}).")
        else:
            print("PyPDF2 extracted empty or whitespace text.")
            # Consider if None is better here for clear failure
            # Keeping '' for consistency with potential combination logic
        return text.strip() # Return stripped text here


    except Exception as e:
        print(f"Error during PyPDF2 extraction for {os.path.basename(file_path)}: {e}", file=sys.stderr)
        return "" # Return empty string on error

def extract_text_from_ocr(file_path):
    """Extract text from a PDF file using OCR (pytesseract)."""
    print(f"Attempting OCR extraction for: {os.path.basename(file_path)}")
    try:
        # This step requires Poppler to be installed on your system
        # You might need to adjust the dpi for better results/performance trade-off
        images = convert_from_path(file_path, dpi=200) # Increased DPI slightly for potentially better OCR
        print(f"Converted PDF to {len(images)} images.")
        text = ""
        for i, image in enumerate(images):
            try:
                page_text = pytesseract.image_to_string(image)
                if page_text:
                     text += page_text + "\n" # Add newline between page texts
                else:
                    print(f"  OCR: No text extracted from image {i + 1}", file=sys.stderr)
            except Exception as img_e:
                 print(f"  Error processing image {i + 1} with Tesseract: {img_e}", file=sys.stderr)
                 pass # Continue to next image

        if text and text.strip():
            print(f"OCR extracted text (approx. length: {len(text.strip())}).")
        else:
            print("OCR extracted empty or whitespace text.")
             # Consider if None is better here for clear failure
        return text.strip() # Return stripped text here

    except Exception as e:
        print(f"Error during OCR extraction for {os.path.basename(file_path)}: {e}", file=sys.stderr)
        if "tesseract is not installed or not in your PATH" in str(e):
             print("Hint: Make sure Tesseract is installed and accessible (e.g., added to system PATH or tesseract_cmd is set correctly).", file=sys.stderr)
        elif "Unable to get page count" in str(e) or "Error: Invalid PDF file" in str(e):
             print("Hint: Make sure Poppler is installed and accessible (especially pdftoppm utility).", file=sys.stderr)
        return "" # Return empty string on error


def preprocess_text(text):
    """Preprocess text by cleaning and normalizing."""
    if text:
        # Replace multiple newlines with a single space, then multiple spaces with single space
        text = ' '.join(text.splitlines()) # Handle various newline types and replace with space
        text = " ".join(text.split())      # Remove extra spaces

        return text
    return "" # Return empty string for consistency if input is falsy

def extract_company_name_from_text(text):
    """Extract the company name from the text using keywords."""
    keywords = ["Care Health Insurance", "ICICI Lombard", "HDFC Ergo"]
    for keyword in keywords:
        if keyword.lower() in text.lower():
            return keyword
    return "Unknown Company"

def determine_company_name(file_name, extracted_text, fallback_insurance_name):
    """Determine the company name based on text, then file name, then fallback."""
    # 1. Check keywords in the combined extracted text
    company_name = extract_company_name_from_text(extracted_text)
    if company_name != "Unknown Company":
        print(f"Company name found in text: {company_name}")
        return company_name

    # 2. If not found in text, check keywords in the file name
    print(f"Company name not found in text, checking file name: {file_name}")
    if "optima-secure" in file_name.lower():
        company_name = "HDFC Ergo"
    elif "care-supreme" in file_name.lower():
        company_name = "Care Health Insurance"
    elif "activate-booster" in file_name.lower():
        company_name = "ICICI Lombard"
    else:
         company_name = "Unknown Company" # Still unknown after checking filename

    if company_name != "Unknown Company":
         print(f"Company name found in file name: {company_name}")
         return company_name

    # 3. If still not found, use the fallback insurance name (from folder)
    print(f"Company name not found in text or file name, falling back to folder name: {fallback_insurance_name}")
    return fallback_insurance_name


def process_file(file_path, insurance_name):
    """Process a single PDF file and save extracted text with metadata to JSON."""
    file_name = os.path.basename(file_path)
    output_file_base, _ = os.path.splitext(file_name)
    output_json_path = os.path.join(OUTPUT_FOLDER, f"{output_file_base}.json")

    print(f"\n--- Processing file: {file_name} ---")
    print(f"Intended output: {output_json_path}")

    # Extract text using PyPDF2 and OCR
    text_pypdf2 = extract_text_from_pdf(file_path)
    text_ocr = extract_text_from_ocr(file_path)

    # Combine the results from both methods (Script 2's combination logic)
    # Using or '' ensures that if one returns None/empty, the other is used or it defaults to empty
    combined_text = (text_pypdf2 or "") + "\n\n" + (text_ocr or "") # Added double newline for separation
    # Remove leading/trailing whitespace from the combined text *before* preprocessing
    combined_text = combined_text.strip()

    # Preprocess the combined text
    extracted_text = preprocess_text(combined_text)

    if not extracted_text:
        print(f"Skipping '{file_name}': Unable to extract any significant text using PyPDF2 or OCR.")
        return # Stop processing this file

    # Determine company name based on text, file name, and folder name fallback
    company_name = determine_company_name(file_name, extracted_text, insurance_name)

    # Prepare data for JSON
    metadata = {
        "file_name": file_name,
        "company_name": company_name,
        "insurance_name": insurance_name, # This comes from the folder name
        "extracted_text": extracted_text # Store the full preprocessed text
    }

    # Save metadata and text to JSON
    try:
        with open(output_json_path, "w", encoding='utf-8') as f:
            json.dump(metadata, f, indent=4, ensure_ascii=False) # ensure_ascii=False for non-ASCII chars
        print(f"Successfully extracted text and saved to {output_json_path}")
    except Exception as e:
        print(f"Error saving JSON file for {file_name} to {output_json_path}: {e}", file=sys.stderr)

    print(f"--- Finished processing file: {file_name} ---\n")


def process_folder(folder_path):
     """Process the brochure and policy wording PDFs in a folder."""
     insurance_name = os.path.basename(folder_path) # Use folder name as insurance name
     print(f"\n--- Processing folder: {folder_path} (Insurance Name: {insurance_name}) ---")

     if not os.path.isdir(folder_path):
         print(f"Error: Folder path does not exist or is not a directory: {folder_path}", file=sys.stderr)
         return

     file_list = os.listdir(folder_path)
     pdf_files = [f for f in file_list if f.lower().endswith(".pdf")]
     print(f"Found {len(pdf_files)} PDF files in the folder.")

     if not pdf_files:
         print("No PDF files found in this folder.")
         return

     # Script 2's original logic classified files and processed specific types.
     # This updated version processes ALL pdfs in the folder, similar to Script 1's loop,
     # but keeps the 'insurance_name' derived from the folder.
     # If you *only* want brochure/policy wording, you'd add the classification logic back here
     # before calling process_file. For now, processing all .pdf:

     for file_name in pdf_files:
         file_path = os.path.join(folder_path, file_name)
         if os.path.isfile(file_path): # Ensure it's a file
              process_file(file_path, insurance_name)
         else:
              print(f"Skipping '{file_name}' as it is not a file (e.g., it's a subfolder).")


def process_all_folders(parent_folder):
    """Process all subfolders in the parent folder."""
    print(f"Starting processing of all folders in: {parent_folder}")

    if not os.path.isdir(parent_folder):
        print(f"Error: Parent folder path does not exist or is not a directory: {parent_folder}", file=sys.stderr)
        return

    # Get all items in the parent folder and filter for directories
    try:
        subfolders = [f.path for f in os.scandir(parent_folder) if f.is_dir()]
    except Exception as e:
        print(f"Error listing contents of {parent_folder}: {e}", file=sys.stderr)
        return

    print(f"Found {len(subfolders)} subfolders to process.")

    if not subfolders:
        print("No subfolders found in the parent directory.")
        return

    for subfolder in subfolders:
        # The processing logic for each subfolder is now encapsulated in process_folder
        process_folder(subfolder)

    print("\n--- Finished processing all folders ---")


# Main execution block
if __name__ == "__main__":
    print(f"Starting script execution with parent folder: {INPUT_PARENT_FOLDER}")
    process_all_folders(INPUT_PARENT_FOLDER)
    print("\nScript finished. Check the output folder for JSON files.")